# Task 1.1: Core Contribution / Architecture

**Paper:** Improving the Fisher Kernel for Large-Scale Image Classification  
**Authors:** Perronnin, Sánchez, Mensink (ECCV 2010)  
**Student:** Prince Sahoo (230060)

## Step-by-Step Method Description

So the idea of this paper is straightforward — you know how in standard image classification, we usually extract visual words from an image using k-means (the Bag-of-Visual-Words approach) and then throw a non-linear SVM at it? That works fine, but non-linear SVMs become impossibly slow when you have hundreds of thousands of training images. This paper takes a different route: instead of making the classifier fancier, they make the image *representation* so powerful that a simple, fast linear SVM can do the job.

Here's how the whole pipeline works, step by step:

- **Step 1:** Local Descriptor Extraction
  - The image is densely sampled with 32×32 pixel patches on a regular grid (every 16 pixels) at five scales. Each patch gives a 128-D SIFT descriptor capturing local gradient orientations. Optionally, 96-D color descriptors (mean + std of R, G, B in 4×4 sub-regions) are also computed.
  - Reference: Section 4.1 (Experimental Setup)
  - This gives us a variable-length set of local features $X = \{x_t, t = 1 \ldots T\}$ — the raw input that the rest of the pipeline will compress into a fixed-length vector.

- **Step 2:** PCA Dimensionality Reduction
  - All descriptors (128-D SIFT or 96-D color) are projected down to 64 dimensions using PCA, learned offline from a large pool of descriptors.
  - Reference: Section 4.1 — *"Both SIFT and color features are reduced to 64 dimensions using PCA."*
  - This is important because the Fisher Vector's dimensionality is $2KD$ — with $K=256$ Gaussians, going from $D=128$ to $D=64$ cuts the final representation from 65,536 to 32,768 dimensions. It also decorrelates the features, which helps since the GMM in the next step uses diagonal covariances.

- **Step 3:** GMM Training (Visual Vocabulary)
  - A Gaussian Mixture Model with $K=256$ components is trained offline via Expectation-Maximization (EM) on descriptors sampled from many images. Each Gaussian has a weight $w_i$, a mean $\mu_i$, and a diagonal covariance $\sigma_i^2$. Think of this as learning a "soft" visual vocabulary — instead of hard-assigning each patch to its nearest cluster center like k-means, the GMM gives a probability distribution over all 256 "visual words" for each descriptor.
  - Reference: Section 2 — the GMM is defined as $u_\lambda(x) = \sum_{i=1}^{K} w_i u_i(x)$. Equation 6 gives the soft-assignment (posterior) $\gamma_t(i) = \frac{w_i u_i(x_t)}{\sum_j w_j u_j(x_t)}$.
  - The GMM captures what "typical" image content looks like across all images. The Fisher Vector will then measure how a specific image *differs* from this average.

- **Step 4:** Fisher Vector Computation
  - For each image, we compute how its descriptors deviate from the GMM. Mathematically, we take the gradient of the log-likelihood with respect to the GMM parameters, then whiten it using the Fisher Information Matrix. This gives two vectors per Gaussian:
    - Mean gradient: $\mathcal{G}_{\mu,i}^X = \frac{1}{T\sqrt{w_i}} \sum_t \gamma_t(i) \frac{x_t - \mu_i}{\sigma_i}$ — captures how descriptors assigned to Gaussian $i$ shift away from its mean.
    - Variance gradient: $\mathcal{G}_{\sigma,i}^X = \frac{1}{T\sqrt{2w_i}} \sum_t \gamma_t(i) \left[\frac{(x_t - \mu_i)^2}{\sigma_i^2} - 1\right]$ — captures whether those descriptors are more or less spread out than expected.
  - Reference: Section 2, Equations 7 and 8. The overall Fisher Vector framework comes from Jaakkola & Haussler [21], applied to images by Perronnin & Dance [22].
  - We concatenate all these vectors for all $K$ Gaussians, getting a $2KD = 32{,}768$-dimensional fixed-length vector. Unlike BOV counts, this encodes *how* the image's patches differ from each visual word — the direction and magnitude of the deviation. This is the core representation that bridges generative (GMM) and discriminative (SVM) approaches.

- **Step 5:** Power Normalization *(this paper's key contribution #1)*
  - Each dimension $z$ of the Fisher Vector is transformed element-wise as: $f(z) = \text{sign}(z) \cdot |z|^{0.5}$. It's basically a signed square root.
  - Reference: Section 3.2, Equation 15, and Figure 1. The authors tried other functions like $\text{sign}(z)\log(1+\alpha|z|)$ and $\text{asinh}(\alpha z)$ but the power function worked best.
  - Here's the problem it solves: with 256 Gaussians, most descriptors get assigned to only a handful of them ($\gamma_t(i) \approx 0$ for most $i$). So the Fisher Vector becomes extremely sparse — most values are basically zero (see Figure 1c). A linear SVM uses dot-products, and dot-products are terrible at measuring similarity between sparse vectors. Rather than switching to a non-linear kernel (which kills scalability), power normalization "un-sparsifies" the distribution (Figure 1d). This alone boosts AP by +6.3% on PASCAL VOC 2007 — the biggest single improvement in the paper.

- **Step 6:** L2 Normalization *(this paper's key contribution #2)*
  - The power-normalized Fisher Vector is then divided by its L2 norm to produce a unit-length vector.
  - Reference: Section 3.1, Equations 9–14. The math shows that the Fisher Vector can be written as $\mathcal{G}_\lambda^X \approx \omega \cdot (\text{image-specific gradient})$ where $\omega$ is the fraction of the image that contains the object vs. generic background (Equation 13). The background part vanishes because the GMM was trained to maximize its likelihood (Equation 12).
  - The issue is that $\omega$ varies between images — a zoomed-in photo of a cat has $\omega \approx 1$ while a distant cat in a landscape has $\omega \approx 0.1$. Without L2 normalization, these two images of the same cat would look very different to the classifier just because of the background amount. L2 normalization cancels out $\omega$, making the representation focus purely on what makes the image distinctive. Adds +3.9% AP.

- **Step 7:** Spatial Pyramid Matching *(this paper's contribution #3)*
  - Instead of computing one Fisher Vector for the whole image, we split the image into 8 regions: the full image, 3 horizontal strips (top/middle/bottom), and 4 quadrants. We compute a separate Fisher Vector for each region, normalize each one independently (power then L2), and concatenate them.
  - Reference: Section 3.3. The spatial split follows the PASCAL VOC 2008 winning strategy [9]. Average pooling is used since it's the natural mechanism for Fisher Vectors (Equations 7–8).
  - Standard Fisher Vectors throw away all spatial information — a plane in the sky and a plane on the ground look the same. Spatial pyramids add back some rough layout. Also, sub-regions have even fewer descriptors, making sparsity worse, so power normalization is especially helpful here. Adds +2.4% AP.

- **Step 8:** Linear SVM Classification
  - One-vs-rest linear SVMs are trained on the final normalized vectors. The optimizer is SGD in the primal (the Pegasos algorithm [12]), with the regularization parameter tuned on a validation set. For multi-channel setups (SIFT + color), two classifiers are trained separately and their scores averaged.
  - Reference: Section 4.1 — *"We learn linear SVMs with a hinge loss using the primal formulation and SGD."* They also tried logistic regression but it was 2× slower with no improvement.
  - This is the payoff. Linear SVMs cost $O(N)$ to train (vs. $O(N^2)$–$O(N^3)$ for RBF-SVMs). The whole point of Steps 5–7 is to make the representation good enough that a linear classifier is all you need. The paper shows the entire system (350K training images) can be trained and evaluated in under a day on a single CPU.

---
### Final Summary

This paper solves the problem of making image classification both accurate and scalable to hundreds of thousands of images; the authors claim that their approach is superior because by applying power normalization, L2 normalization, and spatial pyramids to the Fisher Vector from a GMM, they create a representation so expressive that a simple, fast linear SVM can match or exceed the accuracy of expensive non-linear systems — achieving 58.3% AP on PASCAL VOC 2007 using only SIFT and a linear classifier.